# HTML Quick-Styler
Run this cell first to install the required packages on your Colab Environment.

In [ ]:
!pip install gradio transformers torch accelerate


## Run the Application
Run the cell below. A public Gradio link (`https://xxxx.gradio.live`) will appear at the bottom!

In [ ]:
import gradio as gr
import json
import re
from datetime import datetime
from typing import Tuple, Dict, List
import random
# Option to use dummy AI if running locally without GPU/Transformers
import os

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

class ColorPaletteGenerator:
    """Generate visually cohesive color palettes"""
    
    @staticmethod
    def generate_palette(style: str = "modern") -> Dict[str, str]:
        """Generate a cohesive color palette based on style"""
        
        palettes = {
            'modern': {
                'primary': '#667eea',
                'secondary': '#764ba2',
                'accent': '#f093fb',
                'background': '#0f172a',
                'surface': '#1e293b',
                'text_primary': '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border': '#334155'
            },
            'vibrant': {
                'primary': '#ff6b6b',
                'secondary': '#ff8e72',
                'accent': '#feca57',
                'background': '#111111',
                'surface': '#222222',
                'text_primary': '#ffffff',
                'text_secondary': '#cccccc',
                'border': '#444444'
            },
             'ocean': {
                'primary': '#009ffd',
                'secondary': '#2a2a72',
                'accent': '#00c3ff',
                'background': '#0b132b',
                'surface': '#1c2541',
                'text_primary': '#e0fbfc',
                'text_secondary': '#98c1d9',
                'border': '#3a506b'
            },
            'forest': {
                'primary': '#2d6a4f',
                'secondary': '#1b4332',
                'accent': '#52b788',
                'background': '#081c15',
                'surface': '#143601',
                'text_primary': '#d8f3dc',
                'text_secondary': '#95d5b2',
                'border': '#40916c'
            },
            'sunset': {
                'primary': '#ff7b54',
                'secondary': '#ffb26b',
                'accent': '#ffd56b',
                'background': '#1a0b0b',
                'surface': '#2d1414',
                'text_primary': '#ffffff',
                'text_secondary': '#ffdfdf',
                'border': '#5a2e2e'
            },
            'luxury': {
                'primary': '#d4af37',
                'secondary': '#aa8000',
                'accent': '#f3e5ab',
                'background': '#000000',
                'surface': '#1a1a1a',
                'text_primary': '#ffffff',
                'text_secondary': '#b3b3b3',
                'border': '#333333'
            },
            'creative': {
                'primary': '#9b5de5',
                'secondary': '#f15bb5',
                'accent': '#fee440',
                'background': '#00bbf9',
                'surface': '#00f5d4',
                'text_primary': '#ffffff',
                'text_secondary': '#e0e0e0',
                'border': '#00a3cc'
            },
            'minimal': {
                'primary': '#2c3e50',
                'secondary': '#34495e',
                'accent': '#e74c3c',
                'background': '#ffffff',
                'surface': '#f8f9fa',
                'text_primary': '#212529',
                'text_secondary': '#6c757d',
                'border': '#dee2e6'
            }
        }
        
        return palettes.get(style.lower(), palettes['modern'])

class HTMLPageBuilder:
    """Build styled HTML pages from descriptions"""
    
    @staticmethod
    def extract_sections(description: str) -> List[str]:
        """Extract sections from description"""
        sections = []
        
        section_keywords = {
            'hero': ['hero', 'banner', 'landing'],
            'features': ['features', 'services', 'benefits'],
            'about': ['about', 'team', 'company'],
            'portfolio': ['portfolio', 'gallery', 'work', 'projects'],
            'testimonials': ['testimonial', 'reviews', 'feedback'],
            'pricing': ['pricing', 'plans', 'cost'],
            'contact': ['contact', 'form', 'reach'],
            'footer': ['footer', 'bottom']
        }
        
        desc_lower = description.lower()
        
        for section_type, keywords in section_keywords.items():
            for keyword in keywords:
                if keyword in desc_lower:
                    sections.append(section_type)
                    break
                    
        if not sections:
            sections = ['hero', 'features', 'contact', 'footer']
            
        return sections
        
    @staticmethod
    def generate_meta_tags(title: str, description: str) -> str:
        return f"""
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
    <meta name="description" content="{description}">
    <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css" rel="stylesheet">
"""

    @staticmethod
    def generate_css(palette: Dict) -> str:
        """Generate comprehensive CSS"""
        css = f'''    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}

        :root {{
            --primary: {palette['primary']};
            --secondary: {palette['secondary']};
            --accent: {palette['accent']};
            --background: {palette['background']};
            --surface: {palette['surface']};
            --text-primary: {palette['text_primary']};
            --text-secondary: {palette['text_secondary']};
            --border: {palette['border']};
        }}

        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: var(--background);
            color: var(--text-primary);
            line-height: 1.6;
        }}

        /* Typography */
        h1, h2, h3, h4, h5, h6 {{
            color: var(--text-primary);
            margin-bottom: 1rem;
        }}

        p {{
            color: var(--text-secondary);
            margin-bottom: 1.5rem;
        }}

        /* Buttons */
        .btn {{
            display: inline-block;
            padding: 0.8rem 1.5rem;
            border-radius: 5px;
            text-decoration: none;
            font-weight: 600;
            transition: all 0.3s ease;
            cursor: pointer;
            border: none;
        }}

        .btn-primary {{
            background: linear-gradient(135deg, var(--primary), var(--secondary));
            color: white;
            box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);
        }}

        .btn-primary:hover {{
            transform: translateY(-2px);
            box-shadow: 0 6px 20px rgba(0, 0, 0, 0.3);
        }}

        /* Sections */
        section {{
            padding: 5rem 2rem;
        }}

        .container {{
            max-width: 1200px;
            margin: 0 auto;
        }}

        .section-title {{
            text-align: center;
            font-size: 2.5rem;
            margin-bottom: 3rem;
            position: relative;
        }}

        .section-title::after {{
            content: '';
            display: block;
            width: 50px;
            height: 3px;
            background: var(--accent);
            margin: 1rem auto 0;
        }}

        /* Component Styles (Hero, Features, etc.) */
        .hero {{
            min-height: 80vh;
            display: flex;
            align-items: center;
            justify-content: center;
            text-align: center;
            background: linear-gradient(rgba(0,0,0,0.7), rgba(0,0,0,0.7)), url('https://source.unsplash.com/random/1920x1080/?technology,abstract') center/cover;
            color: white;
            padding: 2rem;
        }}

        .hero h1 {{
            font-size: 4rem;
            margin-bottom: 1.5rem;
            color: white;
        }}

        .hero p {{
            font-size: 1.2rem;
            max-width: 600px;
            margin: 0 auto 2rem;
            color: rgba(255,255,255,0.8);
        }}

        .grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
            gap: 2rem;
        }}

        .card {{
            background: var(--surface);
            padding: 2rem;
            border-radius: 10px;
            border: 1px solid var(--border);
            transition: transform 0.3s ease;
        }}

        .card:hover {{
            transform: translateY(-5px);
            border-color: var(--accent);
        }}

        .card-icon {{
            font-size: 2.5rem;
            color: var(--accent);
            margin-bottom: 1.5rem;
        }}

        /* Footer */
        footer {{
            background: var(--surface);
            padding: 3rem 2rem;
            text-align: center;
            border-top: 1px solid var(--border);
        }}

        /* Responsive */
        @media (max-width: 768px) {{
            .hero h1 {{
                font-size: 2.5rem;
            }}
            section {{
                padding: 3rem 1rem;
            }}
        }}
    </style>'''
        return css

    @staticmethod
    def generate_section_html(section: str, palette: Dict) -> str:
        """Generate HTML for a specific section"""
        if section == 'hero':
            return """
    <section class="hero">
        <div class="container">
            <h1>Welcome to Our Platform</h1>
            <p>Discover innovation and excellence. We build digital solutions that transform businesses and elevate user experiences.</p>
            <a href="#contact" class="btn btn-primary">Get Started Today</a>
        </div>
    </section>"""
        
        elif section == 'features':
            return """
    <section id="features" class="container">
        <h2 class="section-title">Our Features</h2>
        <div class="grid">
            <div class="card">
                <i class="fas fa-rocket card-icon"></i>
                <h3>Fast Performance</h3>
                <p>Lightning quick load times and optimized rendering for the best user experience.</p>
            </div>
            <div class="card">
                <i class="fas fa-shield-alt card-icon"></i>
                <h3>High Security</h3>
                <p>Enterprise-grade security protocols to keep your data safe and protected.</p>
            </div>
            <div class="card">
                <i class="fas fa-mobile-alt card-icon"></i>
                <h3>Responsive Design</h3>
                <p>Looks perfect on all devices, from large desktop screens to mobile phones.</p>
            </div>
        </div>
    </section>"""
        
        elif section == 'about':
             return """
    <section id="about" style="background: var(--surface);">
        <div class="container grid" style="align-items: center;">
            <div>
                <h2 style="margin-bottom: 1rem; font-size: 2.5rem;">About Us</h2>
                <div style="width: 50px; height: 3px; background: var(--accent); margin-bottom: 1.5rem;"></div>
                <p>We are a team of passionate developers, designers, and strategists dedicated to creating exceptional digital products. With years of industry experience, we understand what it takes to succeed in the modern digital landscape.</p>
                <p>Our mission is to empower businesses with cutting-edge technology and beautiful design.</p>
            </div>
            <div style="background: var(--border); height: 300px; border-radius: 10px; display: flex; align-items: center; justify-content: center;">
                <i class="fas fa-users" style="font-size: 4rem; color: var(--text-secondary);"></i>
            </div>
        </div>
    </section>"""

        elif section == 'portfolio':
             return """
    <section id="portfolio" class="container">
        <h2 class="section-title">Our Work</h2>
        <div class="grid">
            <div class="card" style="padding: 0; overflow: hidden;">
                <div style="height: 200px; background: var(--border); display: flex; align-items: center; justify-content: center;">Project Image</div>
                <div style="padding: 1.5rem;">
                    <h3>E-Commerce Platform</h3>
                    <p style="margin-bottom: 0;">A full-featured online store.</p>
                </div>
            </div>
             <div class="card" style="padding: 0; overflow: hidden;">
                <div style="height: 200px; background: var(--border); display: flex; align-items: center; justify-content: center;">Project Image</div>
                <div style="padding: 1.5rem;">
                    <h3>SaaS Dashboard</h3>
                    <p style="margin-bottom: 0;">Analytics and management tool.</p>
                </div>
            </div>
            <div class="card" style="padding: 0; overflow: hidden;">
                <div style="height: 200px; background: var(--border); display: flex; align-items: center; justify-content: center;">Project Image</div>
                <div style="padding: 1.5rem;">
                    <h3>Corporate Website</h3>
                    <p style="margin-bottom: 0;">Brand identity and information.</p>
                </div>
            </div>
        </div>
    </section>"""

        elif section == 'contact':
            return """
    <section id="contact" style="background: var(--surface);">
        <div class="container">
            <h2 class="section-title">Contact Us</h2>
            <div style="max-width: 600px; margin: 0 auto; background: var(--background); padding: 2rem; border-radius: 10px; border: 1px solid var(--border);">
                <form>
                    <div style="margin-bottom: 1.5rem;">
                        <label style="display: block; margin-bottom: 0.5rem;">Name</label>
                        <input type="text" style="width: 100%; padding: 0.8rem; background: var(--surface); border: 1px solid var(--border); color: var(--text-primary); border-radius: 5px;">
                    </div>
                    <div style="margin-bottom: 1.5rem;">
                        <label style="display: block; margin-bottom: 0.5rem;">Email</label>
                        <input type="email" style="width: 100%; padding: 0.8rem; background: var(--surface); border: 1px solid var(--border); color: var(--text-primary); border-radius: 5px;">
                    </div>
                    <div style="margin-bottom: 1.5rem;">
                        <label style="display: block; margin-bottom: 0.5rem;">Message</label>
                        <textarea rows="4" style="width: 100%; padding: 0.8rem; background: var(--surface); border: 1px solid var(--border); color: var(--text-primary); border-radius: 5px; resize: vertical;"></textarea>
                    </div>
                    <button type="button" class="btn btn-primary" style="width: 100%;">Send Message</button>
                </form>
            </div>
        </div>
    </section>"""
        
        elif section == 'footer':
            return """
    <footer>
        <div class="container">
            <h3 style="margin-bottom: 1rem;">BrandLogo</h3>
            <p style="margin-bottom: 2rem; max-width: 400px; margin-left: auto; margin-right: auto;">Creating digital experiences that matter.</p>
            <div style="margin-bottom: 2rem;">
                <a href="#" style="color: var(--text-secondary); margin: 0 10px; font-size: 1.5rem;"><i class="fab fa-twitter"></i></a>
                <a href="#" style="color: var(--text-secondary); margin: 0 10px; font-size: 1.5rem;"><i class="fab fa-linkedin"></i></a>
                <a href="#" style="color: var(--text-secondary); margin: 0 10px; font-size: 1.5rem;"><i class="fab fa-github"></i></a>
            </div>
            <p style="margin-bottom: 0; font-size: 0.9rem;">&copy; 2026 Your Company. All rights reserved.</p>
        </div>
    </footer>"""
            
        return ""

class HTMLGenerationEngine:
    """Main engine for HTML page generation"""
    
    # Initialize Granite model (if available)
    tokenizer = None
    model = None
    
    if HAS_TRANSFORMERS and os.environ.get('USE_GPU', '0') == '1':
        print("Loading Granite Model...")
        try:
            tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
            model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
        except Exception as e:
            print(f"Failed to load model: {e}")

    @staticmethod
    def generate_ai_content(prompt: str) -> str:
        """Call Granite model for content suggestion if loaded"""
        if HTMLGenerationEngine.model is None or HTMLGenerationEngine.tokenizer is None:
            # Fallback to standard content if no model
            return "AI content generation is currently disabled. Using standard templates."
            
        try:
             messages = [
                {"role": "user", "content": prompt},
             ]
             inputs = HTMLGenerationEngine.tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt"
             ).to(HTMLGenerationEngine.model.device)
             
             outputs = HTMLGenerationEngine.model.generate(**inputs, max_new_tokens=40)
             text = HTMLGenerationEngine.tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])
             return text
        except Exception as e:
             return f"Error computing: {e}"

    
    @staticmethod
    def generate_page(title: str, description: str, style: str = "modern") -> str:
        """Generate complete HTML page"""
        
        # Generate color palette
        palette = ColorPaletteGenerator.generate_palette(style)
        
        # Extract sections
        sections = HTMLPageBuilder.extract_sections(description)
        
        # Generate meta tags
        meta_tags = HTMLPageBuilder.generate_meta_tags(title, description)
        
        # Generate CSS
        css = HTMLPageBuilder.generate_css(palette)
        
        # Generate sections
        sections_html = '\n'.join([
            HTMLPageBuilder.generate_section_html(section, palette)
            for section in sections
        ])
        
        # Combine
        full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
{meta_tags}
{css}
</head>
<body>
{sections_html}
</body>
</html>"""

        return full_html


# --- Gradio UI ---

def create_ui():
    custom_css = """
    .dark footer { color: white !important; opacity: 1 !important; }
    .dark footer a { color: #60a5fa !important; opacity: 1 !important; text-decoration: underline; }
    .dark .gradio-container .prose p, .dark .gradio-container .prose h1, .dark .gradio-container .prose h2, .dark .gradio-container .prose h3 { color: white !important; }
    .dark .gradio-container footer .meta-text { color: white !important; opacity: 1 !important; }
    
    /* Standard Light Theme Improvements */
    footer { color: #333 !important; }
    footer a { color: #2563eb !important; }
    """
    with gr.Blocks(theme=gr.themes.Monochrome(), css=custom_css) as demo:
        with gr.Row():
            with gr.Column(scale=4):
                gr.Markdown("# 🧬 HTML Quick-Styler")
                gr.Markdown("### AI-Powered HTML Page Generator with IBM Granite 3.3 2B\n*v4.0 ADVANCED*")
            with gr.Column(scale=1):
                theme_toggle = gr.Radio(["☀️", "🌙"], label="Display Theme", value="🌙", interactive=True)
        
        # Session state for generated HTML
        current_html = gr.State("")
        
        # JS for theme switching
        dark_js = """
        (theme) => {
            if (theme.includes('🌙')) {
                document.body.classList.add('dark');
                document.getElementsByClassName('gradio-container')[0].classList.add('dark');
            } else {
                document.body.classList.remove('dark');
                document.getElementsByClassName('gradio-container')[0].classList.remove('dark');
            }
        }
        """
        theme_toggle.change(None, inputs=[theme_toggle], js=dark_js)
        
        with gr.Tabs() as main_tabs:
            def go_home():
                return gr.Tabs(selected="tab_generate")

            # ============ TAB 1: Generate Page ============
            with gr.TabItem("⚡ Generate", id="tab_generate"):
                with gr.Row():
                    # Left Column: Inputs
                    with gr.Column(scale=1):
                        gr.Markdown("### ✨ Page Details")
                        
                        page_title = gr.Textbox(
                            label="Page Title",
                            value="Modern Project",
                            lines=1
                        )
                        
                        page_description = gr.Textbox(
                            label="Page Description & Sections",
                            value="hero, features, contact, footer",
                            lines=4
                        )
                        
                        color_style = gr.Dropdown(
                            choices=["modern", "vibrant", "ocean", "forest", "sunset", "luxury", "creative", "minimal"],
                            label="Style",
                            value="modern"
                        )
                        
                        gr.Markdown("**Detected Sections**")
                        detected_sections_display = gr.HTML("<div style='display:flex;gap:10px;'><span style='background:#333;color:white;padding:5px 10px;border-radius:15px;font-size:0.8rem;'>hero</span><span style='background:#333;color:white;padding:5px 10px;border-radius:15px;font-size:0.8rem;'>features</span><span style='background:#333;color:white;padding:5px 10px;border-radius:15px;font-size:0.8rem;'>contact</span><span style='background:#333;color:white;padding:5px 10px;border-radius:15px;font-size:0.8rem;'>footer</span></div>")
                        
                        generate_btn = gr.Button("🚀 Generate Production Page", size="lg", variant="primary")
                    
                    # Right Column: Outputs
                    with gr.Column(scale=2):
                        gr.Markdown("### 💻 Generated Code")
                        
                        with gr.Tabs():
                            with gr.TabItem("CODE VIEW"):
                                code_output = gr.Code(label="Index.html", language="html", interactive=False)
                            
                            with gr.TabItem("INLINE PREVIEW"):
                                html_output = gr.HTML(label="Live Preview")
                        
                        status_box = gr.Textbox(label="Status", value="Ready", interactive=False)
                        export_btn = gr.Button("📦 Export .ZIP (Mock)", variant="primary")
            
            # ============ TAB 2: Live Preview (Full) ============
            with gr.TabItem("👁️ Full Screen Preview"):
                 with gr.Row():
                     back_btn_preview = gr.Button("⬅️ Back to Home", size="sm")
                     full_preview_btn = gr.Button("🔄 Refresh Preview", size="sm")
                 
                 back_btn_preview.click(go_home, outputs=main_tabs)
                 full_preview_html = gr.HTML("""<p style='text-align:center; color:#777; padding:40px;'>No preview loaded. Generate a page first.</p>""")
            
            # ============ TAB 3: Color Palettes ============
            with gr.TabItem("🎨 Color Palettes"):
                back_btn_palettes = gr.Button("⬅️ Back to Home", size="sm")
                back_btn_palettes.click(go_home, outputs=main_tabs)
                
                gr.Markdown("### Explore Color Schemes")
                with gr.Row():
                   with gr.Column():
                        style_selector = gr.Radio(
                            ["modern", "vibrant", "ocean", "forest", "sunset", "luxury", "creative", "minimal"],
                            value="modern",
                            label="Choose Theme Style"
                        )
                        palette_btn = gr.Button("Get Palette Colors", size="lg", variant="primary")
                   with gr.Column():
                        palette_display = gr.JSON(label="Palette Data")
                        
                   def show_palette(style_name):
                       return ColorPaletteGenerator.generate_palette(style_name)
                       
                   palette_btn.click(show_palette, inputs=style_selector, outputs=palette_display)

            # Main Generation Logic (Moved up to support templates)
            def on_generate(title, desc, style):
                # 1. Update detected UI badges based on desc
                sections = HTMLPageBuilder.extract_sections(desc)
                badges_html = "<div style='display:flex;flex-wrap:wrap;gap:10px;'>"
                for s in sections:
                     badges_html += f"<span style='background:#3b82f6;color:white;padding:5px 10px;border-radius:15px;font-size:0.8rem;'>{s}</span>"
                badges_html += "</div>"
                
                # 2. Build HTML
                html_content = HTMLGenerationEngine.generate_page(title, desc, style)
                
                # 3. Format outputs (Code, HTML render, state, status, badges)
                return html_content, f"<div style='border:1px solid #333; height: 500px; overflow-y: scroll;'>{html_content}</div>", html_content, "✅ Success! HTML generated successfully.", badges_html

            def load_template_with_preview(t_title, t_desc, t_style):
                tab_update = gr.Tabs(selected="tab_generate")
                code, preview, state, status, badges = on_generate(t_title, t_desc, t_style)
                return tab_update, t_title, t_desc, t_style, code, preview, state, status, badges

            # ============ TAB 4: Templates ============
            with gr.TabItem("📝 Templates"):
                back_btn_templates = gr.Button("⬅️ Back to Home", size="sm")
                back_btn_templates.click(go_home, outputs=main_tabs)
                
                gr.Markdown("### Quick Templates")
                
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("#### Portfolio Website\nHero, Portfolio Gallery, About, Contact, Footer")
                        portfolio_btn = gr.Button("Use Template", size="sm")
                        
                        def load_portfolio():
                            return load_template_with_preview("John Doe Portfolio", "I am a designer. hero banner, show my portfolio gallery with 6 items, about section with my bio, contact form to reach me, and a footer.", "creative")
                            
                        portfolio_btn.click(load_portfolio, outputs=[main_tabs, page_title, page_description, color_style, code_output, html_output, current_html, status_box, detected_sections_display])
                        
                    with gr.Column():
                        gr.Markdown("#### SaaS Landing Page\nHero, Features, Pricing, Testimonials, Contact, Footer")
                        saas_btn = gr.Button("Use Template", size="sm")
                        
                        def load_saas():
                            return load_template_with_preview("TechFlow SaaS", "hero banner for our software, check our features and benefits, pricing plans, customer testimonials, and contact form, and footer.", "modern")
                            
                        saas_btn.click(load_saas, outputs=[main_tabs, page_title, page_description, color_style, code_output, html_output, current_html, status_box, detected_sections_display])
                        
                    with gr.Column():
                        gr.Markdown("#### Corporate Website\nHero, About, Services, Team, Contact, Footer")
                        corp_btn = gr.Button("Use Template", size="sm")
                        
                        def load_corp():
                             return load_template_with_preview("Global Corp Inc", "hero, read about our corporate history and team, see the features and services we offer, contact form, footer.", "luxury")
                             
                        corp_btn.click(load_corp, outputs=[main_tabs, page_title, page_description, color_style, code_output, html_output, current_html, status_box, detected_sections_display])

                with gr.Row():
                    with gr.Column():
                        gr.Markdown("#### E-Commerce Store\nHero, Features, Pricing, Contact, Footer")
                        ecommerce_btn = gr.Button("Use Template", size="sm")
                        
                        def load_ecommerce():
                            return load_template_with_preview("Style E-Commerce", "hero banner for sale, features of our products, pricing plans, contact form, footer", "vibrant")
                            
                        ecommerce_btn.click(load_ecommerce, outputs=[main_tabs, page_title, page_description, color_style, code_output, html_output, current_html, status_box, detected_sections_display])
                        
                    with gr.Column():
                        gr.Markdown("#### Blog / Magazine\nHero, About, Portfolio, Contact, Footer")
                        blog_btn = gr.Button("Use Template", size="sm")
                        
                        def load_blog():
                            return load_template_with_preview("Tech Blog", "hero banner, reading articles, about authors, contact form, footer", "minimal")
                            
                        blog_btn.click(load_blog, outputs=[main_tabs, page_title, page_description, color_style, code_output, html_output, current_html, status_box, detected_sections_display])
                        
                    with gr.Column():
                        gr.Markdown("#### Creative Agency\nHero, About, Services, Portfolio, Contact, Footer")
                        agency_btn = gr.Button("Use Template", size="sm")
                        
                        def load_agency():
                             return load_template_with_preview("Creative Agency", "hero banner, about our creative team, services we offer, gallery of past work, contact form, footer", "creative")
                             
                        agency_btn.click(load_agency, outputs=[main_tabs, page_title, page_description, color_style, code_output, html_output, current_html, status_box, detected_sections_display])


            # ============ TAB 5: Guide ============
            with gr.TabItem("📚 Guide"):
                back_btn_guide = gr.Button("⬅️ Back to Home", size="sm")
                back_btn_guide.click(go_home, outputs=main_tabs)
                
                gr.Markdown("""
                ### HTML Quick-Styler Guide
                
                #### 1. Enter Details
                - **Title:** The web page `<title>`.
                - **Description:** A text description of what you want. The tool will scan your prompt for section keywords.
                - **Style:** Pick from 8 predefined aesthetic color palettes.
                
                #### 2. Automatic Section Detection
                The engine recognizes these keywords in your description:
                - `hero` / `banner` / `landing` -> **Hero Section**
                - `features` / `services` / `benefits` -> **Features Grid**
                - `about` / `team` / `company` -> **About Section**
                - `portfolio` / `gallery` / `projects` -> **Portfolio Section**
                - `testimonials` / `reviews` -> **Testimonial Section**
                - `pricing` / `plans` -> **Pricing Section**
                - `contact` / `form` -> **Contact Form**
                - `footer` / `bottom` -> **Footer**
                
                #### 3. Google Colab & IBM Granite Setup
                When running in Google Colab, if the transformers library is detected and memory provides, the IBM Granite model will spin up. To enable:
                `!pip install transformers accelerate torch gradio`
                Ensure you have a GPU environment (like T4) enabled in Colab!
                """)

        generate_btn.click(
            on_generate, 
            inputs=[page_title, page_description, color_style],
            outputs=[code_output, html_output, current_html, status_box, detected_sections_display]
        )
        
        # Live Preview Refresh Logic
        def refresh_full_preview(html_state):
             if not html_state:
                 return "<p style='text-align:center; color:#777; padding:40px;'>No preview loaded. Generate a page first.</p>"
             return f"<div style='background: white; width: 100%; height: 80vh; overflow-y: scroll; border-radius: 8px;'>{html_state}</div>"

        full_preview_btn.click(
             refresh_full_preview,
             inputs=[current_html],
             outputs=[full_preview_html]
        )

    return demo

if __name__ == "__main__":
    app = create_ui()
    
    print("\n✅ HTML Quick-Styler Advanced Ready!\n")
    # Using share=True as specified in requirements for Colab deployment
    app.launch(share=True, show_error=True, show_api=False)
